# 01 · Data Ingestion

Downloads / reads all raw data sources and writes parquet files to `data/raw/`.

**Sources**
| Source | Access | Output |
|--------|--------|--------|
| IOU Interconnection Apps | 5 CSVs in `data/raw/interconnection/` | `dgstats_raw.parquet` |
| CAISO load | `gridstatus` API | aggregated to `caiso_monthly_merged.parquet` |
| CEC monthly generation | Excel files in `data/raw/cec/` | merged into `caiso_monthly_merged.parquet` |
| NREL DeepSolar CA | CSV | `deepsolar_ca_tracts.parquet` |
| Census ZIP–Tract crosswalk | XLSX | `zip_tract_xwalk.parquet` |
| Utility territories | Shapefile in `data/external/` | `utility_territories.geojson` |

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
EXT = ROOT / 'data' / 'external'
RAW.mkdir(parents=True, exist_ok=True)

import pandas as pd
from data_ingestion import (
    load_interconnection_apps,
    load_utility_territories,
    load_cec_monthly_generation,
    fetch_caiso_all,
    load_deepsolar,
    load_zip_tract_crosswalk,
)

## 1.1 IOU Interconnection Applications (replaces CEC DGStats)

Reads the 5 IOU interconnection CSVs from `data/raw/interconnection/` (PG&E historical + current, SCE historical + current, SDG&E combined). Filters to residential NEM/NBT photovoltaic installs with valid CA ZIP codes, approved 2015–2023.

In [2]:
interconnect_dir = RAW / 'interconnection'
assert interconnect_dir.exists(), f'Expected CSV directory at {interconnect_dir}'

dgstats = load_interconnection_apps(interconnect_dir)
print(f'Residential NEM installs loaded: {len(dgstats):,}')
dgstats.head(3)

Residential NEM installs loaded: 1,503,615


,application_id,utility,zip_code,system_size_dc,battery_storage,sector,app_approved_date,cost,program_type
842,PGE-INT-113272533,PG&E,94062,5.652,0.0,Residential,2015-06-15,NaN,NEM
1259,PGE-INT-113272960,PG&E,95404,5.220,0.0,Residential,2015-09-17,NaN,NEM
2043,PGE-INT-113273778,PG&E,94549,15.268,0.0,Residential,2015-03-11,NaN,NEM


In [3]:
dgstats.to_parquet(RAW / 'dgstats_raw.parquet', index=False)
print('Saved dgstats_raw.parquet')

Saved dgstats_raw.parquet


## 1.2 CAISO Grid Data (load via gridstatus + CEC monthly generation)

2-phase fetch — runs once, results cached in `data/raw/caiso_shards/`:
- **Phase 1** (~2 min): 108 monthly load calls via `gridstatus.get_load_hourly` → aggregated to monthly GWh
- **Phase 2** (seconds): reads CEC monthly electricity generation Excel files from `data/raw/cec/`
  - **Pre-requisite**: download the CEC "Monthly Electricity Generation by Energy Source" Excel file(s)
    from https://www.energy.ca.gov/data-reports/energy-almanac and place in `data/raw/cec/`
- **Phase 3**: left-join on (year, month), compute `net_load_gwh`, `net_load_ratio`, `monthly_ramp_gwh`
- **Phase 4**: QC row-count + null checks

Output columns: `year`, `month`, `demand_gwh`, `solar_gwh`, `wind_gwh`, `total_generation_gwh`,
`net_load_gwh`, `net_load_ratio`, `monthly_ramp_gwh`

In [4]:
caiso = fetch_caiso_all(start_year=2015, end_year=2023, data_dir=ROOT / 'data')
print(f'CAISO monthly rows: {len(caiso):,}  |  expected 108 (9 years × 12 months)')
print(caiso.dtypes)
caiso.head(3)

CAISO monthly rows: 108  |  expected 108 (9 years × 12 months)
year                      int32
month                     int32
demand_gwh              float64
solar_gwh               float64
wind_gwh                float64
total_generation_gwh    float64
net_load_gwh            float64
net_load_ratio          float64
monthly_ramp_gwh        float64
dtype: object


,year,month,demand_gwh,solar_gwh,wind_gwh,total_generation_gwh,net_load_gwh,net_load_ratio,monthly_ramp_gwh
0,2015,1,17803.362,1253.833333,1015.0,16349.583333,15534.528667,0.872562,NaN
1,2015,2,15997.661,1253.833333,1015.0,16349.583333,13728.827667,0.858177,1805.701
2,2015,3,18211.601,1253.833333,1015.0,16349.583333,15942.767667,0.875418,2213.940


In [5]:
caiso.to_parquet(RAW / 'caiso_monthly_raw.parquet', index=False)
print('Saved caiso_monthly_raw.parquet')

Saved caiso_monthly_raw.parquet


## 1.3 NREL DeepSolar CA

Place `deepsolar_ca_tracts.csv` in `data/raw/` (available from Kaggle or the project's GitHub).

In [6]:
ds_csv = RAW / 'deepsolar_ca_tracts.csv'
assert ds_csv.exists(), f'Place DeepSolar CSV at {ds_csv}'

deepsolar = load_deepsolar(ds_csv)
print(f'DeepSolar CA tracts: {len(deepsolar):,}')
deepsolar.to_parquet(RAW / 'deepsolar_ca_tracts.parquet', index=False)

DeepSolar CA tracts: 8,056


## 1.4 ZIP–Tract Crosswalk

HUD ZIP-to-Tract crosswalk (Dec 2023 vintage, XLSX) is already at `data/raw/zip_tract_xwalk.xlsx`.

In [7]:
xwalk_path = RAW / 'zip_tract_xwalk.xlsx'
assert xwalk_path.exists(), f'Expected crosswalk at {xwalk_path}'

xwalk = load_zip_tract_crosswalk(xwalk_path)
print(f'ZIP-tract pairs (CA): {len(xwalk):,}')
xwalk.to_parquet(RAW / 'zip_tract_xwalk.parquet', index=False)
print('Saved zip_tract_xwalk.parquet')

ZIP-tract pairs (CA): 16,022
Saved zip_tract_xwalk.parquet


## Summary

| File | Description |
|------|-------------|
| `dgstats_raw.parquet` | Residential NEM installs, 2015–2023, CA IOUs |
| `caiso_monthly_raw.parquet` | CAISO monthly grid data (load + CEC generation), 2015–2023 (108 rows) |
| `deepsolar_ca_tracts.parquet` | NREL DeepSolar CA census tracts |
| `zip_tract_xwalk.parquet` | HUD ZIP-to-tract crosswalk (CA only) |
| `data/external/utility_territories.geojson` | PG&E / SCE / SDG&E service area polygons |

→ Proceed to `02_cleaning_qa.ipynb`

In [8]:
util_shp = EXT / 'ElectricLoadServingEntities_IOU_POU.shp'
assert util_shp.exists(), f'Expected shapefile at {util_shp}'

util_gdf = load_utility_territories(util_shp)
print(f'Utility territories loaded: {len(util_gdf)} features')
print(util_gdf[['Acronym', 'Utility', 'Type']].to_string(index=False))

util_gdf.to_file(EXT / 'utility_territories.geojson', driver='GeoJSON')
print('Saved utility_territories.geojson')

Utility territories loaded: 3 features
Acronym                        Utility Type
    SCE     Southern California Edison  IOU
   PG&E Pacific Gas & Electric Company  IOU
  SDG&E       San Diego Gas & Electric  IOU


Saved utility_territories.geojson


## Summary

| File | Rows |
|------|------|
| `dgstats_raw.parquet` | — |
| `caiso_hourly_raw.parquet` | — |
| `deepsolar_ca_tracts.parquet` | — |
| `zip_tract_xwalk.parquet` | — |

→ Proceed to `02_cleaning_qa.ipynb`